# Lab 40 (solution): Annotation quality and the judge ceiling

Reference implementation. [Lab 38](../../38-calibrating-the-eval-gate/) validated the judge against **one** annotator. This lab adds **three** and measures inter-annotator agreement (Fleiss' kappa) — the ceiling on how good any judge can look — then scores the judge against the human consensus and against that ceiling.

Ships `multi_annotator_labels.jsonl` (three annotators over Lab 38's 24 candidates).

## Step 0: Setup + judge

In [ ]:
import json
import os
import pathlib
import re
from dotenv import load_dotenv
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break
PROVIDER="openai"
JUDGE_MODEL={"openai":"gpt-4o","anthropic":"claude-opus-4-8"}[PROVIDER]
print(f"judge={JUDGE_MODEL}")

In [ ]:
def chat(messages, model, temperature=0.0):
    if PROVIDER=="openai":
        from openai import OpenAI
        r=OpenAI().chat.completions.create(model=model,messages=messages,temperature=temperature)
        return r.choices[0].message.content or ""
    from anthropic import Anthropic
    system=next((m["content"] for m in messages if m["role"]=="system"),"")
    ns=[m for m in messages if m["role"]!="system"]
    r=Anthropic().messages.create(model=model,system=system,messages=ns,max_tokens=400,temperature=temperature)
    return "".join(b.text for b in r.content if hasattr(b,"text"))
def parse_judge(raw):
    raw=re.sub(r"^```(json)?|```$","",raw.strip(),flags=re.MULTILINE).strip()
    try:
        obj = json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        obj = json.loads(m.group(0)) if m else {}
    return {"correct":bool(obj.get("correct",False))}
RUBRIC=('Judge the candidate against the reference. JSON only: {"correct": true/false}. '
        "correct = conveys the reference facts (paraphrase ok; vague/evasive = not correct).")
def llm_judge(query, candidate, reference):
    raw=chat([{"role":"system","content":RUBRIC},
              {"role":"user","content":f"Q: {query}\nReference: {reference}\nCandidate: {candidate}"}],model=JUDGE_MODEL)
    return parse_judge(raw)["correct"]

## Step 1: Multiple annotators on the same items

In [ ]:
# Lab 38 used one annotator. A single annotator hides a hard truth: humans disagree
# on the borderline cases, and that disagreement is the CEILING on how good any judge
# can look. Here three annotators labeled the same 24 candidates (1=correct, 0=not).
with open("./multi_annotator_labels.jsonl") as f:
    multi = [json.loads(line) for line in f]
# The query text lives in Lab 38's file; join on id.
with open("../38-calibrating-the-eval-gate/judge_validation.jsonl") as f:
    base_rows = [json.loads(line) for line in f]
base = {row["id"]: row for row in base_rows}
for r in multi:
    r["query"] = base[r["id"]]["query"]
print(f"{len(multi)} items x 3 annotators")
splits=[r["id"] for r in multi if len({r["a1"],r["a2"],r["a3"]})>1]
print(f"annotators disagree on {len(splits)}: {splits}")

## Step 2: Inter-annotator agreement (Fleiss' kappa)

The ceiling.

In [ ]:
def fleiss_kappa(count_matrix):
    """count_matrix[i] = [n raters choosing cat0, ... cat_{k-1}] for item i (n per item)."""
    N = len(count_matrix)
    n = sum(count_matrix[0])
    k = len(count_matrix[0])
    P=[(sum(c*c for c in row)-n)/(n*(n-1)) for row in count_matrix]   # per-item agreement
    Pbar=sum(P)/N
    pj=[sum(row[j] for row in count_matrix)/(N*n) for j in range(k)]  # category marginals
    Pe=sum(p*p for p in pj)
    return (Pbar-Pe)/(1-Pe)

counts=[[ [r["a1"],r["a2"],r["a3"]].count(0), [r["a1"],r["a2"],r["a3"]].count(1) ] for r in multi]
kf=fleiss_kappa(counts)
print(f"Fleiss' kappa (inter-annotator agreement): {kf:.2f}")
print("Landis-Koch: 0.61-0.80 substantial, 0.81+ almost perfect.")

## Step 3: Pairwise agreement

Who diverges, and why that points at the guidelines.

In [ ]:
from sklearn.metrics import cohen_kappa_score
A = [r["a1"] for r in multi]
B = [r["a2"] for r in multi]
C = [r["a3"] for r in multi]
print("pairwise Cohen's kappa (find the divergent annotator):")
print(f"  a1-a2: {cohen_kappa_score(A,B):.2f}")
print(f"  a1-a3: {cohen_kappa_score(A,C):.2f}")
print(f"  a2-a3: {cohen_kappa_score(B,C):.2f}")
print("A low pair points at an annotator using a different rubric in their head - a")
print("signal to re-train annotators or sharpen the guidelines, not to blame the judge.")

## Step 4: Consensus + clear vs ambiguous

In [ ]:
# Consensus label = majority vote. Items with full agreement are 'clear'; split items
# are inherently ambiguous and should be reported separately, not used to fail a judge.
consensus=[1 if (r["a1"]+r["a2"]+r["a3"])>=2 else 0 for r in multi]
clear_idx=[i for i,r in enumerate(multi) if len({r["a1"],r["a2"],r["a3"]})==1]
ambig_idx=[i for i,r in enumerate(multi) if i not in clear_idx]
print(f"{len(clear_idx)} unanimous (clear) items, {len(ambig_idx)} split (ambiguous) items")

## Step 5: Judge vs the human ceiling

In [ ]:
from sklearn.metrics import accuracy_score
# Run the judge, score it against the consensus, and compare to the human ceiling.
judge=[]
for r in multi:
    ref=("This question is unanswerable; abstaining is correct."
         if r["candidate"].strip().upper().startswith("INSUFFICIENT") else r["note"])
    judge.append(1 if llm_judge(r["query"], r["candidate"], ref) else 0)

kjc=cohen_kappa_score(consensus, judge)
print(f"human inter-annotator agreement (Fleiss): {kf:.2f}   <- the CEILING")
print(f"judge-vs-consensus (Cohen):               {kjc:.2f}")
if kjc <= kf+0.05:
    print("\nThe judge is at or near the human ceiling. You cannot meaningfully push a judge")
    print("past the agreement humans reach with each other - beyond that you are fitting")
    print("annotation noise. Report the judge against this ceiling, not against 1.0.")
acc_clear=accuracy_score([consensus[i] for i in clear_idx],[judge[i] for i in clear_idx])
acc_ambig=accuracy_score([consensus[i] for i in ambig_idx],[judge[i] for i in ambig_idx]) if ambig_idx else float("nan")
print(f"\njudge accuracy on CLEAR items:     {acc_clear:.2f}  (should be near 1.0)")
print(f"judge accuracy on AMBIGUOUS items: {acc_ambig:.2f}  (humans split here too - report separately)")

## Step 6: What multiple annotators change

In [ ]:
# What changes once you have multiple annotators:
#  - You get a CEILING (inter-annotator agreement). A judge near it is as good as the
#    task allows; a judge far below it has real room to improve.
#  - You can separate the judge's errors on clear items (real bugs) from disagreement on
#    ambiguous items (where even humans split - not the judge's fault).
#  - A divergent annotator pair tells you to fix the GUIDELINES, often a bigger lever
#    than tuning the judge.
print("Inter-annotator agreement is the yardstick. Measure the judge against the human")
print("ceiling and against the clear-item subset - not against an impossible 1.0.")

## What you built

A multi-annotator validation set, inter-annotator agreement (Fleiss' kappa) as the ceiling on judge quality, pairwise agreement to find the divergent annotator, and a judge scored against the human consensus and reported separately on clear vs ambiguous items. The takeaway: a judge near the human ceiling is as good as the task allows; the items humans split on are not the judge's failures.

**Where this simplifies:** three annotators on 24 items is small — real sets use more annotators and items, and you would also track each annotator's agreement with consensus over time to catch drift; Fleiss' kappa assumes a fixed number of raters per item (use Krippendorff's alpha when raters or items are missing); the kappa magnitude depends on label prevalence, so report it with the confusion, not alone.

This feeds [Lab 38](../../38-calibrating-the-eval-gate/): trust judged metrics only up to the human ceiling you measured here.